# Sewage Defect Detection — Tier 2 Colab Training

Boost baseline YOLO26m (mAP@0.5 = 0.44) using:
- imgsz = 1280 + `yolo26-p2.yaml` (small-object P2 head)
- copy_paste / mosaic / mixup augmentation, cosine LR, 300 epochs
- Existing class importance weights (Crack 2.6, Hole 2.4, Utility 2.8, ...)
- SAHI sliced inference sweep (256/384/512 px tiles)

**Runtime**: GPU (T4 free / A100 Pro). Confirm `Runtime → Change runtime type → GPU` before running.

## 1. GPU / environment check

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv
import torch
print('torch:', torch.__version__, '| cuda:', torch.cuda.is_available(),
      '| device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')

## 2. Install dependencies

In [ ]:
!pip install -q -U ultralytics sahi pycocotools pyyaml

## 3. Mount Drive and stage the dataset

Adjust `DRIVE_DATASET_ROOT` to wherever you stored the Roboflow `sewage-yolo26` export in your Drive.
Expected layout: `sewage-yolo26/{train,valid,test}/{images,labels}` + `data.yaml`.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_DATASET_ROOT = '/content/drive/MyDrive/cnn-assignment3/sewage-yolo26'  # EDIT ME
WORKSPACE = '/content/cnn-assignment3'

import os, shutil, pathlib
os.makedirs(f'{WORKSPACE}/src/data', exist_ok=True)
target = f'{WORKSPACE}/src/data/sewage-yolo26'
if not os.path.exists(target):
    print('Copying dataset from Drive — this can take a few minutes...')
    shutil.copytree(DRIVE_DATASET_ROOT, target)
print('Dataset ready:', target)
!ls {target}

## 4. Fix dataset paths if needed

Roboflow exports often use `../train/images` style paths that resolve relative to the YAML file. This cell rewrites them to the unambiguous form so Ultralytics picks the right images on Colab.

In [ ]:
import yaml, pathlib
data_yaml = f'{WORKSPACE}/src/data/sewage-yolo26/data.yaml'
with open(data_yaml) as f:
    cfg = yaml.safe_load(f)
cfg['path'] = f'{WORKSPACE}/src/data/sewage-yolo26'
cfg['train'] = 'train/images'
cfg['val'] = 'valid/images'
cfg['test'] = 'test/images'
with open(data_yaml, 'w') as f:
    yaml.safe_dump(cfg, f, sort_keys=False)
print(open(data_yaml).read())

## 5. Tier 2 training (P2 head + imgsz=1280 + copy-paste + cosine LR)

On T4 (16 GiB), `batch=4` is the safe default at imgsz=1280 with the P2 head.
On A100 (40 GiB) you can push `batch=12` or higher.
Set `EPOCHS=300` with `patience=50` to let cosine LR + early stopping decide when to stop.

In [ ]:
from ultralytics import YOLO

# Start from yolo26-p2 architecture, load pretrained yolo26m weights via .load()
model = YOLO('yolo26-p2.yaml')
try:
    model.load('yolo26m.pt')
    print('Loaded yolo26m.pt pretrained weights into P2 architecture.')
except Exception as exc:
    print('Pretrained transfer failed:', exc, '— training from scratch.')

CLASS_IMPORTANCE_WEIGHTS = {
    'Buckling': 1.8, 'Crack': 2.6, 'Debris': 1.0,
    'Hole': 2.4, 'Joint offset': 1.7, 'Obstacle': 1.2, 'Utility intrusion': 2.8,
}
class_names = list(cfg['names']) if isinstance(cfg['names'], list) else [cfg['names'][i] for i in sorted(cfg['names'])]
cls_weights = [CLASS_IMPORTANCE_WEIGHTS.get(n, 1.0) for n in class_names]

train_kwargs = dict(
    data=data_yaml,
    epochs=300,
    imgsz=1280,
    batch=4,                # bump to 12 on A100
    device=0,
    optimizer='SGD',         # MuSGD if your ultralytics version supports it
    cos_lr=True,
    patience=50,
    plots=True,
    project='runs/detect', name='tier2_p2_1280',
    # Augmentation
    mosaic=1.0, close_mosaic=20,
    mixup=0.10, copy_paste=0.30,
    scale=0.6, degrees=10.0, translate=0.10,
    hsv_v=0.30, hsv_s=0.60, fliplr=0.5,
)
try:
    train_kwargs['cls_weights'] = cls_weights
    results = model.train(**train_kwargs)
except TypeError:
    # Older ultralytics — cls_weights not exposed.
    train_kwargs.pop('cls_weights', None)
    results = model.train(**train_kwargs)

## 6. Standard validation on test split

In [ ]:
best_pt = 'runs/detect/tier2_p2_1280/weights/best.pt'
best = YOLO(best_pt)
test_metrics = best.val(data=data_yaml, split='test', imgsz=1280, device=0,
                       conf=0.001, iou=0.6, augment=True, plots=True)
print(f'mAP@0.5      = {test_metrics.box.map50:.4f}')
print(f'mAP@0.5:0.95 = {test_metrics.box.map:.4f}')
print(f'Precision    = {test_metrics.box.mp:.4f}')
print(f'Recall       = {test_metrics.box.mr:.4f}')

## 7. SAHI slice-size sweep (Tier 1 on top of Tier 2 model)

Runs standard inference + 256/384/512 tile sweep with COCO-style mAP. Picks the tile size that maximises mAP_small without hurting mAP_large.

In [ ]:
import sys, importlib
sys.path.insert(0, f'{WORKSPACE}/src')

# Bring the local sahi_inference.py into Colab.
# It expects DATA_YAML_PATH relative to cwd; chdir to workspace first.
import os
os.chdir(WORKSPACE)

import sahi_inference
importlib.reload(sahi_inference)
sahi_inference.WEIGHTS_PATH = best_pt
sahi_inference.EVAL_SPLIT = 'test'
sahi_inference.SLICE_SIZE_SWEEP_ENABLE = True
sahi_inference.SLICE_SIZE_SWEEP_CANDIDATES = (256, 384, 512)
sahi_inference.RUN_STANDARD_BASELINE = True
sahi_inference.main()

## 8. Save artifacts back to Drive

Copies the best checkpoint, training plots, and validation summary so you can rerun SAHI / further experiments without retraining.

In [ ]:
import shutil, time
stamp = time.strftime('%Y%m%d-%H%M')
drive_out = f'/content/drive/MyDrive/cnn-assignment3/runs/tier2_p2_1280_{stamp}'
shutil.copytree('runs/detect/tier2_p2_1280', drive_out)
print('Saved to', drive_out)

## 9. Optional — Tier 3: sliced training data (run after Tier 2 if time allows)

Pre-slices the train split with SAHI so the next training run sees tile-scale crops in addition to full frames. Often adds another 3-8 pp on small-class mAP.

In [ ]:
# Convert YOLO to COCO
!sahi coco yolov5 \
  --image_dir src/data/sewage-yolo26/train/images \
  --label_dir src/data/sewage-yolo26/train/labels \
  --output_dir src/data/sewage-yolo26-coco/train

# Slice the COCO dataset to 512 px tiles, 20% overlap
!sahi coco slice \
  --coco_path src/data/sewage-yolo26-coco/train/result.json \
  --image_dir src/data/sewage-yolo26/train/images \
  --slice_size 512 --overlap_ratio 0.2 \
  --output_dir src/data/sewage-yolo26-sliced/train

# After this, merge sliced + original into a new data.yaml and rerun cell 5.
print('Sliced training data ready under src/data/sewage-yolo26-sliced/train')